In [1]:
from pathlib import Path

import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

from plant_shape_analysis import PlantSequencesDataset, LeafSequencesDataset
from plant_shape_analysis.vis.plot_functions import visualize_plant_sequence, visualize_leaf_sequence

%load_ext autoreload
%autoreload 2

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
selected_sequences = [
"maize_control_plant1",
"maize_control_plant2",
"maize_control_plant3",
"sorghum_control_plant1",
"sorghum_highlight_plant2",
"sorghum_highlight_plant3",
"tobacco_control_plant1",
"tobacco_control_plant2",
"tobacco_shade_plant3",
"tomato1_control2_plant2",
"tomato1_heat_plant3",
"tomato2_control_plant3"
]

### Visualize aligned plant sequences

In [5]:
dataset_path = Path("../data/TrackPlant3D/versions")
plant_dataset = PlantSequencesDataset(
    dataset_path,
    use_ply=True,
    alignment_method="stem_based",
    estimate_normals=True,
    auto_download=False
)

Applying STEM_BASED alignment to 43 plant sequences...
STEM_BASED alignment complete. 43 sequences aligned.
Estimating normals for all sparse plants in the dataset...
Normal estimation complete.


Plot one plant example with normals

In [ ]:
sample_scan = plant_dataset[0]["timepoints"][0]
points = sample_scan["points"]
normals = sample_scan["normals"]

# Plot
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.normals = o3d.utility.Vector3dVector(normals)
# pcd.normalize_normals() # they're already normalized
scale = 0.5  # Adjust this number — smaller = shorter, larger = longer
pcd.normals = o3d.utility.Vector3dVector(
    [n * scale for n in pcd.normals]
)
o3d.visualization.draw_geometries(
        [pcd],
        point_show_normal=True,
        up=[0, 0, 1],
        front=[1, 0, 0],
        lookat=[0, 0, 0],
    )

Visualize a whole sequence with normals

In [14]:
plant_ts = plant_dataset.get_timeseries_by_sequence_name("maize_control_plant1")
visualize_plant_sequence(plant_ts, window_name=f"{plant_ts['sequence_name']}", spacing=100, normals_scale=0.2)

Visualize all selected plant sequences with initial normals

In [ ]:
for plant_ts in plant_dataset:
    if plant_ts["sequence_name"] in selected_sequences:
        visualize_plant_sequence(plant_ts, window_name=f"{plant_ts['sequence_name']}", spacing=100, normals_scale=0.2)
    break

In [ ]:
# # Save ply files for fast debugging wrt to dataset v2 (end of notebook)
# for plant_ts in plant_dataset:
#     sequence_name = plant_ts["sequence_name"]
#     sequence_path = dataset_path / "v2_debug_original_plant_normals" / sequence_name
#     sequence_path.mkdir(parents=True, exist_ok=True)

#     for timepoint in plant_ts["timepoints"]:
#         timepoint_name = timepoint["day"]
#         points = timepoint["points"]
#         normals = timepoint["normals"]
#         labels = timepoint["labels"]
#         leaf_tip_idxs = timepoint["leaf_tip_idxs"]

#         # Create o3d.t point cloud
#         pcd = o3d.t.geometry.PointCloud(points)
#         pcd.point["normals"] = o3d.core.Tensor(
#             normals.astype(np.float32), dtype=o3d.core.Dtype.Float32
#         )
#         pcd.point["organ_label"] = o3d.core.Tensor(
#         labels.reshape(-1, 1), dtype=o3d.core.Dtype.Int32
#         )
#         leaf_tip_mask = np.zeros(len(points), dtype=np.uint8)
#         if len(leaf_tip_idxs) > 0:
#             leaf_tip_mask[leaf_tip_idxs] = 1
#         pcd.point["is_leaf_tip"] = o3d.core.Tensor(
#             leaf_tip_mask.reshape(-1, 1), dtype=o3d.core.Dtype.UInt8
#         )

#         # Save PLY
#         o3d.t.io.write_point_cloud(
#             str(sequence_path / f"{timepoint_name}.ply"),
#             pcd,
#         )

### Fix leaves alignment and make normals consistent across timepoints

In [ ]:
dataset_path = Path("../data/TrackPlant3D/versions")
plant_dataset = PlantSequencesDataset(
    dataset_path,
    use_ply=True,
    alignment_method="stem_based",
    estimate_normals=False,
    auto_download=False
)

In [ ]:
sample_seq = plant_dataset.get_timeseries_by_sequence_name("maize_control_plant1")
sample_seq["timepoints"]

In [ ]:
dataset_path = Path("../data/TrackPlant3D/versions")
leaf_dataset = LeafSequencesDataset(
    dataset_path,
    use_ply=True,
    apply_alignment=True,
    plant_alignment_method="stem_based",
    estimate_plant_normals=True,
    auto_download=False
)

In [ ]:
# TODO: fix visualization function needing transformations key-value pair of leafs
sample_seq = leaf_dataset.get_timeseries_by_sequence_name("maize_control_plant1_leaf1") 

In [ ]:
# for time in sample_seq["timepoints"]:
#     print(sample_seq["sequence_name"], sample_seq["leaf_id"])
#     print(time["day"])
#     output_folder = Path(f"../tmp/{sample_seq['sequence_name']}")
#     output_folder.mkdir(parents=True, exist_ok=True)
#     pcd = o3d.t.geometry.PointCloud(time["points"])

#     o3d.t.io.write_point_cloud(str(output_folder / f"day{time['day']}.ply"), pcd)

In [ ]:
# Visualize leaf sequence
visualize_leaf_sequence(sample_seq, show_coordinate_frame=True, show_principal_axes=True)

In [ ]:
# Visualize all leaves of selected sequences
for plant_name in selected_sequences:
    all_leaves = leaf_dataset.get_timeseries_by_plant_sequence(plant_name)
    for leaf_seq in all_leaves:
        visualize_leaf_sequence(leaf_seq, window_name=f"{leaf_seq['sequence_name']}_leaf{leaf_seq['leaf_id']}", show_coordinate_frame=True)


#### Analyze individual alignment steps

In [ ]:
aligned_timepoints = leaf_dataset._prealign_with_leaf_tips(sample_seq['timepoints'])
aligned_seq = {
    "timepoints": aligned_timepoints,
    "sequence_name": sample_seq["sequence_name"],
    "leaf_id": sample_seq["leaf_id"]
}
visualize_leaf_sequence(aligned_seq, show_coordinate_frame=True)

In [ ]:
aligned_timepoints = leaf_dataset._align_leaves_to_xy_plane(sample_seq['timepoints'])
aligned_seq = {
    "timepoints": aligned_timepoints,
    "sequence_name": sample_seq["sequence_name"],
    "leaf_id": sample_seq["leaf_id"]
}
visualize_leaf_sequence(aligned_seq, show_coordinate_frame=True)

In [ ]:
# It seems this works but only on unaligned leaves
aligned_timepoints_consistent = leaf_dataset._enforce_temporal_normal_consistency(aligned_seq["timepoints"])
aligned_seq_consistent = {
    "timepoints": aligned_timepoints_consistent,
    "sequence_name": sample_seq["sequence_name"],
    "leaf_id": sample_seq["leaf_id"]
}
visualize_leaf_sequence(aligned_seq_consistent, show_coordinate_frame=True)

In [ ]:
def estimate_normals(sample_seq):
    for timepoint in sample_seq["timepoints"]:
        points = timepoint["points"]
        if points is not None and len(points) > 0:
            # Estimate normals using Open3D
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(points)
            pcd.estimate_normals()
            pcd.orient_normals_to_align_with_direction()
            pcd.orient_normals_consistent_tangent_plane(k=30)
            normals = np.asarray(pcd.normals)
            timepoint["normals"] = normals

# Estimate normals for each timepoint
#estimate_normals(sample_seq)

# Ensure temporal consistency of normals across timepoints
#leaf_dataset._enforce_temporal_normal_consistency(sample_seq["timepoints"])

# Visualize
visualize_leaf_sequence(sample_seq)

#### Check saved transformations and inverse

In [ ]:
sample_seq_trans = leaf_dataset.transformations['maize_control_plant1_leaf1']
sample_seq_trans

In [ ]:
composed_matrices = LeafSequencesDataset.compose_transformations(sample_seq_trans)
composed_matrices

In [ ]:
original_timepoints = LeafSequencesDataset.apply_inverse_transformations(
    sample_seq,
    sample_seq_trans
)
original_sequence = {
    "sequence_name": sample_seq["sequence_name"],
    "leaf_id": sample_seq["leaf_id"],
    "timepoints": original_timepoints,
}
original_sequence

In [ ]:
visualize_leaf_sequence(original_sequence, show_coordinate_frame=True)

In [ ]:
# # Visualize unaligned leaf sequence (test inverse transformations)
# leaf_dataset2 = LeafSequencesDataset(
#     dataset_path,
#     use_ply=True,
#     apply_alignment=False,
#     plant_alignment_method="stem_based",
#     estimate_plant_normals=True,
#     auto_download=False
# )
# sample_seq2 = leaf_dataset2.get_timeseries_by_sequence_name("maize_control_plant1_leaf1") 
# visualize_leaf_sequence(sample_seq2, show_coordinate_frame=True)

#### Compare leaves to original plant

In [ ]:
# Visualize all leaves of a plant
all_leaves = leaf_dataset.get_timeseries_by_plant_sequence("maize_control_plant1")
len(all_leaves)

In [ ]:
for leaf_seq in all_leaves:
    visualize_leaf_sequence(leaf_seq, window_name=f"{leaf_seq['sequence_name']}_leaf{leaf_seq['leaf_id']}", show_coordinate_frame=True)

In [ ]:
# Inver trasnforms back to original space
inverted_leaves = []
for leaf_ts in all_leaves:
    seq_name = leaf_ts["sequence_name"]
    trans = leaf_dataset.transformations[seq_name]
    # trans = leaf_ts['transformation_stages'] # This also works, NOTE: we're storing the same info twice, at dataset level and at the sequence level
    original_tps = LeafSequencesDataset.apply_inverse_transformations(leaf_ts, trans)

    inverted_leaves.append({
    "sequence_name": seq_name,
    "leaf_id": leaf_ts["leaf_id"],
    "timepoints": original_tps,
    })

In [ ]:
# Visualize inverted leaves
for leaf_seq in inverted_leaves:
    visualize_leaf_sequence(leaf_seq, window_name=f"{leaf_seq['sequence_name']}_leaf{leaf_seq['leaf_id']}_inverted", show_coordinate_frame=True)

In [ ]:
# Plot plant sequence with all leaves in original space
plant_name = "maize_control_plant1"
plant_sample = leaf_dataset.plant_dataset.get_timeseries_by_sequence_name(plant_name)
visualize_plant_sequence(plant_sample, window_name=f"{plant_sample['sequence_name']}_original_space", spacing=100, normals_scale=0.2)

### Final visual checks

We saved v2 of the datset including the previously corrected normals into the plants with `scripts/save_aligned_plants_with_normals.py`

In [19]:
dataset_path = Path("../data/TrackPlant3D/versions")
plant_dataset2 = PlantSequencesDataset(dataset_path, version="v2", use_ply=True, alignment_method=None, estimate_normals=False)

In [23]:
visualize_plant_sequence(
    plant_dataset2.get_timeseries_by_sequence_name("maize_control_plant3"),
    window_name="maize_control_plant1_v2",
    spacing=100,
    normals_scale=0.2
    )

Visualize all selected plant sequences

In [28]:
for seq in selected_sequences:
    visualize_plant_sequence(
        plant_dataset2.get_timeseries_by_sequence_name(seq),
        window_name=seq,
        spacing=100,
        normals_scale=0.2
    )

Remarks on current v2 (WIP):
- maize_control_plant1 OK
- maize_control_plant2 has inconsistent normal face orientation across leaves
- maize_control_plant3 has inconsistent temporal normals
- sorghum_control_plant1 has inconsistent temporal normals
- sorghum_highlight_plant2 has inconsistent normal face orientation across leaves
- sorghum_highlight_plant3 bad alignment and has both inconsistent temporal normals and inconsistent normal face orientation across leaves
- tobacco_control_plant1 has bad alignment and inconsistent temporal normals
- tobacco_control_plant2 has inconsistent normal face orientation across leaves
- tobacco_shade_plant3 has inconsistent temporal normals and inconsistent normal face orientation across leaves
- tomato1_control2_plant2 has inconsistent normal face orientation across leaves (**also it has a lot of holes, might want to select a different sequence**)
- tomato1_heat_plant3 has inconsistent normal face orientation across leaves (**also doesn't grow much, so we might want to select a different sequence**)
- tomato2_control_plant3 has inconsistent temporal normals and inconsistent normal face orientation across leaves (**also note that in timestep 6 a petiole is missing which would make stitching plant back together harder afterr independent organ growth [unless interpolation works well]**)
- **Maize and tomato1 plant have not been corrected with respect to their Z axis up direction**